# ETO-Core: Budget-Constrained Candidate Search

Compare four search strategies under the same budget:
1. **Random Search** — random candidate selection
2. **Static Ranking** — rank by sequence-only model
3. **Genetic Algorithm** — evolutionary optimization
4. **ETO-Guided** — ensemble predictions + uncertainty + novelty

In [ ]:
import os
import sys
import json
import numpy as np
import torch

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    device = 'cuda'
else:
    device = 'cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/ETO'
print(f'Drive mounted: {DRIVE_ROOT}')

In [ ]:
%%bash
cd /content
if [ ! -d 'ETO' ]; then
    git clone https://github.com/jianghu105/ETO.git
else
    cd ETO && git pull
fi

In [ ]:
sys.path.insert(0, '/content/ETO')

DATA_PATH = '/content/ETO/dataset/20230531AptDMC双端-MPO.xlsx'
MODEL_DIR = f'{DRIVE_ROOT}/experiments/etocore_v1'
SEARCH_DIR = f'{DRIVE_ROOT}/experiments/search_v1'
os.makedirs(SEARCH_DIR, exist_ok=True)

BUDGET = 100
N_PARENTS = 500
SEED = 42

print(f'Budget: {BUDGET} candidates')
print(f'Output: {SEARCH_DIR}')

## 1. Load Data and Trained Models

In [ ]:
from data.unified_dataset import build_etocore_dataset
from data.eto_dataset import build_dataloaders
from models.eto_core import build_model

print('Loading data...')
dataset_dict = build_etocore_dataset(DATA_PATH, seed=SEED)
metadata = dataset_dict['metadata']
arrays = dataset_dict['arrays']
splits = dataset_dict['splits']
normalizer = dataset_dict['normalizer']

print(f"  {metadata['n_sequences']} sequences, {metadata['n_families']} families")

In [ ]:
from data.preprocessing import load_real_data
from data.unified_dataset import assign_family_ids

df = load_real_data(DATA_PATH, min_total_count=10, seq_length=36)
all_sequences = df['sequence'].tolist()
all_family_ids = assign_family_ids(all_sequences)

test_seq_indices = set(arrays['seq_idx'][splits['test']].tolist())
train_seq_indices = set(arrays['seq_idx'][splits['train']].tolist())
source_indices = [i for i in range(len(all_sequences))
                  if i not in test_seq_indices]
source_sequences = [all_sequences[i] for i in source_indices]
source_families = all_family_ids[source_indices]

print(f'Source pool: {len(source_sequences)} sequences (excl. test set)')
print(f'  {len(np.unique(source_families))} families')

In [ ]:
print('Loading trained models...')
models = []
for model_type in ['history', 'etocore']:
    ckpt_path = f'{MODEL_DIR}/{model_type}/checkpoint_best.pt'
    if not os.path.exists(ckpt_path):
        print(f'  WARNING: {ckpt_path} not found, skipping {model_type}')
        continue
    model = build_model(model_type)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if 'best_state' in ckpt and ckpt['best_state'] is not None:
        model.load_state_dict(ckpt['best_state'])
    else:
        model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()
    models.append(model)
    print(f'  Loaded {model_type}')

print(f'Ensemble: {len(models)} models')

## 2. Generate Candidate Pool

In [ ]:
from search.candidate_generator import CandidatePool

exclude = set(arrays['seq_idx'][splits['test']].tolist())
exclude_seqs = set(all_sequences[i] for i in exclude)

pool = CandidatePool(source_sequences, source_families,
                     exclude_sequences=exclude_seqs)
n_mutants = pool.generate_mutants(n_parents=N_PARENTS,
                                  mutations_per_parent=1,
                                  max_per_parent=20,
                                  seed=SEED)
print(f'Candidate pool: {pool.n_total} mutants from {N_PARENTS} parents')
print(f'  Available: {pool.n_available}')

## 3. Run Search Strategies

In [ ]:
from search.strategies import (
    random_search, static_ranking_search,
    eto_guided_search, genetic_algorithm_search,
)
import time

all_results = {}

print('=' * 60)
print('RANDOM SEARCH')
pool_random = CandidatePool(source_sequences, source_families,
                            exclude_sequences=exclude_seqs)
pool_random.generate_mutants(n_parents=N_PARENTS, mutations_per_parent=1,
                             max_per_parent=20, seed=SEED)
t0 = time.time()
res = random_search(pool_random, budget=BUDGET, seed=SEED)
all_results['random'] = res.summary(pool_random)
all_results['random']['time_s'] = time.time() - t0
print(f"  Selected: {len(res.selected)}, Families: {all_results['random']['n_unique_families']}")

print('\n' + '=' * 60)
print('STATIC RANKING')
pool_static = CandidatePool(source_sequences, source_families,
                            exclude_sequences=exclude_seqs)
pool_static.generate_mutants(n_parents=N_PARENTS, mutations_per_parent=1,
                             max_per_parent=20, seed=SEED)
t0 = time.time()
res = static_ranking_search(models, pool_static, budget=BUDGET,
                            device=device, batch_size=512)
all_results['static_ranking'] = res.summary(pool_static)
all_results['static_ranking']['time_s'] = time.time() - t0
print(f"  Selected: {len(res.selected)}, Mean score: {all_results['static_ranking']['mean_score']:.4f}")

print('\n' + '=' * 60)
print('ETO-GUIDED SEARCH (full)')
pool_eto = CandidatePool(source_sequences, source_families,
                         exclude_sequences=exclude_seqs)
pool_eto.generate_mutants(n_parents=N_PARENTS, mutations_per_parent=1,
                          max_per_parent=20, seed=SEED)
t0 = time.time()
res = eto_guided_search(models, pool_eto, budget=BUDGET,
                        device=device, beta=0.5, gamma=0.3,
                        batch_size=256, mode='full')
all_results['eto_guided'] = res.summary(pool_eto)
all_results['eto_guided']['time_s'] = time.time() - t0
print(f"  Selected: {len(res.selected)}, Mean score: {all_results['eto_guided']['mean_score']:.4f}")

print('\n' + '=' * 60)
print('GENETIC ALGORITHM')
pool_ga = CandidatePool(source_sequences, source_families,
                        exclude_sequences=exclude_seqs)
pool_ga.generate_mutants(n_parents=N_PARENTS, mutations_per_parent=1,
                         max_per_parent=20, seed=SEED)
t0 = time.time()
res = genetic_algorithm_search(models, pool_ga, budget=BUDGET,
                               device=device, population_size=100,
                               n_generations=30, batch_size=512)
all_results['genetic_algorithm'] = res.summary(pool_ga)
all_results['genetic_algorithm']['time_s'] = time.time() - t0
print(f"  Selected: {len(res.selected)}, Mean score: {all_results['genetic_algorithm']['mean_score']:.4f}")

## 4. Results Summary

In [ ]:
with open(f'{SEARCH_DIR}/search_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('=' * 80)
print('SEARCH COMPARISON')
print('=' * 80)
print(f"{'Strategy':<20} {'Budget':>8} {'Families':>10} {'Mean Score':>12} "
      f"{'Novelty':>10} {'Time(s)':>10}")
print('-' * 80)

for name, r in all_results.items():
    print(f"{name:<20} {r['budget']:>8} {r['n_unique_families']:>10} "
          f"{r['mean_score']:>12.4f} {r['mean_novelty']:>10.4f} "
          f"{r['time_s']:>10.1f}")

print(f"\nResults saved to: {SEARCH_DIR}/")

## 5. Scoring Ablation

In [ ]:
ablation_results = {}

for mode in ['prediction', 'prediction+uncertainty',
             'prediction+novelty', 'full']:
    pool_ab = CandidatePool(source_sequences, source_families,
                            exclude_sequences=exclude_seqs)
    pool_ab.generate_mutants(n_parents=N_PARENTS, mutations_per_parent=1,
                             max_per_parent=20, seed=SEED)
    res = eto_guided_search(models, pool_ab, budget=BUDGET,
                            device=device, beta=0.5, gamma=0.3,
                            batch_size=256, mode=mode)
    ablation_results[mode] = res.summary(pool_ab)
    print(f"  {mode:<30} score={ablation_results[mode]['mean_score']:.4f} "
          f"families={ablation_results[mode]['n_unique_families']}")

with open(f'{SEARCH_DIR}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)